# Segmentation using background subtraction

Use background substraction methods to properly segment the moving objects from their background. Use one of the videos with static camera.

Use the following approaches:
- Accumulated weighted image
- Mixture of Gaussian (MOG2)


In [2]:
import cv2 as cv
import numpy as np
import time

In [3]:
# Video file names
file1 = 'data/crosswalk.avi'
file2 = 'data/fourway.avi'
file3 = 'data/night.avi'

### Accumulated weighted image

In [29]:
# Load the video
cap = cv.VideoCapture(file1)

alpha = 0.02
scale = 0.4
convert_to_gray = True

# Read the first frame
_, frame = cap.read()

# Convert the first frame to grayscale
if convert_to_gray:
    frame = cv.cvtColor(frame, cv.COLOR_BGR2GRAY)

# Initialize the background model
bg_model = np.float32(frame)

while True:
    # Read the next frame of the video
    success, frame = cap.read()
    # Exit the loop if the frame was not successfully read
    if not success:
        cv.waitKey(0)
        break

    # Convert the current frame to grayscale
    if convert_to_gray:
        frame = cv.cvtColor(frame, cv.COLOR_BGR2GRAY)

    # Update the background model using an accumulated weighted image
    cv.accumulateWeighted(frame, bg_model, alpha)

    # Subtract the background
    background = cv.convertScaleAbs(bg_model)
    diff = cv.absdiff(frame, background)
    _, thresh = cv.threshold(diff, 25, 255, cv.THRESH_BINARY)

    if scale != 1:
        frame = cv.resize(frame, (0, 0), fx=scale, fy=scale)
        background = cv.resize(background, (0, 0), fx=scale, fy=scale)
        thresh = cv.resize(thresh, (0, 0), fx=scale, fy=scale)

    # Display the resulting images
    cv.imshow("Frame", frame)
    cv.imshow("Thresh", thresh)
    cv.imshow("Background", background)

    # Exit the program if the user presses the 'q' key
    if cv.waitKey(1) & 0xFF == ord('q'):
        break

# Release the VideoCapture and close all windows
cap.release()
cv.destroyAllWindows()

### Mixture of Gaussian (MOG2)